# LoRA/QLoRA fine-tuning + quantization - Colab runner

Before running anything here:

1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Upload the whole `lora-qlora-pipeline/` folder (the parent of this `notebooks/` folder) to your Google Drive, directly under **My Drive**, so the path is `My Drive/lora-qlora-pipeline`. Easiest way: drag the folder into drive.google.com in your browser. If you're re-uploading after a code update, delete the old copy on Drive first so stale files don't linger.

This notebook mounts Drive and runs each pipeline stage in order. Training checkpoints and all outputs are written straight into the Drive copy, so they survive Colab disconnects - if a session dies mid-training, just reconnect and re-run the training cell; `Trainer` will resume from the latest checkpoint under `outputs/adapters/.../checkpoint-*`.

**Run the cells in order, top to bottom, exactly once each per session.** The install cell below deliberately kills and restarts the Python process straight after installing pinned package versions - Colab keeps old package code loaded in memory otherwise, and a stale `transformers`/`peft` still active in memory is a real, hard-to-diagnose failure mode. When you see the kernel disconnect, that's expected: just continue with the next cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/lora-qlora-pipeline'
assert os.path.isdir(PROJECT_DIR), (
    f"Expected {PROJECT_DIR} to exist. Upload the lora-qlora-pipeline folder "
    "to the root of your Google Drive (My Drive), then re-run this cell."
)
%cd {PROJECT_DIR}

In [ ]:
!pip install -q -r requirements.txt

# Force a clean process restart so the pinned versions just installed are
# actually the ones imported below - Colab does NOT do this automatically,
# and re-running pip install alone is not enough if anything in this kernel
# already imported the old versions.
print("Restarting runtime to load pinned package versions - this is expected, continue with the next cell after it reconnects.")
import os
os.kill(os.getpid(), 9)

**The cell above intentionally crashes the kernel.** Colab will show "Your session crashed" and reconnect automatically within a few seconds - that's the restart working as intended, not a failure. Once it reconnects, run the cell below (which re-mounts Drive, since the previous Python process and its state are gone) and continue down the notebook. Do not re-run the install cell above.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/lora-qlora-pipeline'
%cd {PROJECT_DIR}

import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU visible - set Runtime > Change runtime type > T4 GPU, then re-run this cell.")

import transformers, peft, accelerate, bitsandbytes, datasets as hf_datasets
print("transformers", transformers.__version__, "| peft", peft.__version__, "| accelerate", accelerate.__version__, "| bitsandbytes", bitsandbytes.__version__, "| datasets", hf_datasets.__version__)

## Stage 1: dataset prep

CPU-only, ~a minute or two. Loads `yahma/alpaca-cleaned`, formats through Qwen2.5's chat template, writes `data/alpaca_train.jsonl` and `data/alpaca_val.jsonl`.

In [ ]:
!python src/data.py

## Stage 2: QLoRA fine-tuning

The long one. Loads Qwen2.5-3B-Instruct in 4-bit, attaches LoRA adapters, trains. Watch the printed `trainable params` line early on - it should be well under 1% of total params. Checkpoints save to `outputs/adapters/.../checkpoint-*` on Drive as training progresses.

In [ ]:
!python src/train.py

## Stage 3: evaluation

Compares base vs fine-tuned: masked perplexity on held-out data, plus generations on a fixed prompt set. Writes `outputs/benchmarks/eval_report.json`.

In [ ]:
!python src/evaluate.py

## Stage 4: merge + quantize

Merges the LoRA adapter into a clean fp16 copy of the base model, then clones/builds llama.cpp and produces GGUF files at Q8_0, Q5_K_M, and Q4_K_M. The llama.cpp build step (`cmake --build`) is the slow part the first time - it's skipped on later runs once the binaries exist.

In [ ]:
!apt-get -qq install -y cmake > /dev/null
!python src/quantize.py

## Stage 5: benchmark

Sweeps fp16 / bnb-8bit / bnb-4bit / GGUF Q8_0 / Q5_K_M / Q4_K_M: tokens/sec, memory or file size, and a quality metric for each. Writes `outputs/benchmarks/benchmark_report.json` and prints a summary table.

In [ ]:
!python src/benchmark.py

## Inspect the reports

In [ ]:
import json

with open('outputs/benchmarks/eval_report.json') as f:
    eval_report = json.load(f)
print("Perplexity - base vs fine-tuned:", eval_report['perplexity'])
for item in eval_report['qualitative'][:3]:
    print("\nPrompt:", item['prompt'])
    print("  Base:      ", item['base'][:200])
    print("  Fine-tuned:", item['fine_tuned'][:200])

with open('outputs/benchmarks/benchmark_report.json') as f:
    bench_report = json.load(f)
print("\nHF variants:")
for r in bench_report['hf_variants']:
    print(f"  {r['name']:<10} tok/s={r['tokens_per_sec']:.2f}  vram_gb={r['peak_vram_gb']:.2f}  ppl={r['perplexity']:.3f}")
print("GGUF variants:")
for r in bench_report['gguf_variants']:
    print(f"  {r['name']:<14} tok/s={r['tokens_per_sec']}  size_gb={r['file_size_gb']:.2f}  ppl={r['perplexity_gguf_unmasked']}")